#Notebook 3 - Imputación, Escalamiento y Transformación de "Power Consumption of Tetouan City".

## **1. Introducción**

En este notebook se desarrollan los procesos de imputación, escalamiento y transformación aplicados al dataset Power Consumption of Tetouan City.
Estas etapas permiten mejorar la calidad del dataset, corregir desviaciones introducidas por atípicos o escalas incompatibles y preparar las variables para futuros análisis multivariados o modelos predictivos.

##**2. Imputación**



Antes de aplicar imputación es necesario verificar si existen valores nulos en el dataset.


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Karencgx/Fundamentos_cc_datos/main/datos/Tetuan_City_power_consumption_clean%20(2).csv"
data = pd.read_csv(url)
data.head()


,Day,Hour,Temperature,Humidity,Wind Speed,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,Sunday,0,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386
1,Sunday,0,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434
2,Sunday,0,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373
3,Sunday,0,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711
4,Sunday,0,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964


In [2]:
data.isnull().sum()


,0
Day,0
Hour,0
Temperature,0
Humidity,0
Wind Speed,0
Zone 1 Power Consumption,0
Zone 2 Power Consumption,0
Zone 3 Power Consumption,0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52416 entries, 0 to 52415
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Day                        52416 non-null  object 
 1   Hour                       52416 non-null  int64  
 2   Temperature                52416 non-null  float64
 3   Humidity                   52416 non-null  float64
 4   Wind Speed                 52416 non-null  float64
 5   Zone 1 Power Consumption   52416 non-null  float64
 6   Zone 2  Power Consumption  52416 non-null  float64
 7   Zone 3  Power Consumption  52416 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 3.2+ MB


Los resultados muestran que no existe ningún valor faltante en las 8 variables del dataset.  
Esto significa que no es necesario aplicar técnicas de imputación, ya que la base de datos está completa.

Esto es coherente con el origen del dataset, basado en sensores y registros horarios de consumo energético y clima.

Aun así, esta etapa se documenta para:

- mantener la trazabilidad del pipeline,  
- garantizar el cumplimiento del ciclo completo de ciencia de datos,  
- dejar claridad sobre las decisiones tomadas.

Con esta verificación, procedemos a las etapas de escalamiento y transformación de datos.

##**3. Escalamiento**

En este conjunto de datos no se presentan valores faltantes, por lo que el preprocesamiento se enfoca en la homogeneización de escalas. Las variables numéricas tienen rangos muy distintos (por ejemplo, la humedad puede llegar a 100 %, mientras que la velocidad del viento es menor a 15 m/s), lo cual puede afectar negativamente técnicas posteriores de modelado o análisis multivariado.

Para garantizar que cada variable contribuya de forma equilibrada, se aplica StandardScaler, que transforma cada variable para que tenga:

media = 0

desviación estándar = 1

Este método se selecciona porque:

* Preserva la forma original de la distribución

* Es el más recomendado para modelos lineales y basados en distancia

* Es menos sensible a rangos extremos que MinMaxScaler

* Se usa frecuentemente en problemas reales porque produce resultados estables y consistentes

Además, el objetivo del proyecto no requiere mantener un rango fijo (como 0–1), por lo que StandardScaler es la opción más adecuada.

In [4]:
from sklearn.preprocessing import StandardScaler


In [5]:
numeric_cols = ["Hour", "Temperature", "Humidity", "Wind Speed",
                "Zone 1 Power Consumption", "Zone 2  Power Consumption",
                "Zone 3  Power Consumption"]

scaler = StandardScaler()

data_scaled = data.copy()
data_scaled[numeric_cols] = scaler.fit_transform(data[numeric_cols])

data_scaled.head()

,Day,Hour,Temperature,Humidity,Wind Speed,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,Sunday,-1.661325,-2.106645,0.356277,-0.798900,0.239917,-0.944672,0.363262
1,Sunday,-1.661325,-2.131578,0.401291,-0.798900,-0.354854,-0.320573,0.346669
2,Sunday,-1.661325,-2.148946,0.401291,-0.800178,-0.451143,-0.391398,0.276804
3,Sunday,-1.661325,-2.181962,0.433443,-0.798900,-0.577254,-0.515516,0.160655
4,Sunday,-1.661325,-2.216353,0.478456,-0.799752,-0.702514,-0.609482,0.091663


Después del escalamiento, algunas variables toman valores negativos, lo cual es completamente normal.
Esto ocurre cuando un dato está por debajo de la media de su variable.
Valores positivos indican que están por encima, y valores cercanos a 0 representan valores típicos.

Tras aplicar StandardScaler:

Todas las variables numéricas quedaron normalizadas

Las distribuciones mantienen su forma original, pero ahora con un centro común

El dataset está preparado para análisis y modelado posteriores

In [6]:
data_scaled[numeric_cols].mean().round(4), data_scaled[numeric_cols].std().round(4)


(Hour                        -0.0
 Temperature                  0.0
 Humidity                     0.0
 Wind Speed                  -0.0
 Zone 1 Power Consumption    -0.0
 Zone 2  Power Consumption    0.0
 Zone 3  Power Consumption   -0.0
 dtype: float64,
 Hour                         1.0
 Temperature                  1.0
 Humidity                     1.0
 Wind Speed                   1.0
 Zone 1 Power Consumption     1.0
 Zone 2  Power Consumption    1.0
 Zone 3  Power Consumption    1.0
 dtype: float64)

##**4. Transformación de datos**

La columna Day (Lunes, Martes, etc.) es una variable categórica (una etiqueta, no un número).

Si le diéramos un número cualquiera a cada día (Lunes=1, Martes=2, Miércoles=3, etc.), el modelo de análisis se confundiría, pensando dos cosas incorrectas:

Hay un Orden: Creería que hay un orden o jerarquía. Por ejemplo, que el Sábado (6) es mucho más importante que el Lunes (1), solo porque tiene un número más grande. Un lunes y un sábado son igual de relevantes, solo que con patrones de consumo diferentes.

Relación Lineal: Asumiría que la distancia de Lunes a Martes es la misma que de Miércoles a Jueves.



In [7]:

# La columna 'Day' es categórica nominal y debe transformarse a formato numérico

data_transformed = data_scaled.copy()

# Aplicar One-Hot Encoding a la columna 'Day'
# 'drop_first=True' se usa para evitar la multicolinealidad,
# ya que si Day_Lunes, Day_Martes, ..., Day_Sabado son todos 0, implica que el día es Domingo.
data_transformed = pd.get_dummies(data_transformed, columns=['Day'], drop_first=True, prefix='Day')

print("Variables transformadas (dummies):")
data_transformed.head()
print("\nShape del dataset después de la transformación:", data_transformed.shape)

Variables transformadas (dummies):

Shape del dataset después de la transformación: (52416, 13)


Para evitar que el modelo asuma cosas que no son ciertas, se usa la Codificación One-Hot. Esto es como crear una bandera para cada día:

Se crean columnas nuevas para cada día (Day_Lunes, Day_Martes, Day_Miercoles, etc.).

Si la fila es un Martes, solo la columna Day_Martes se enciende con un 1, y el resto se quedan en 0.

De esta manera, el modelo simplemente ve: "Cuando esta columna es 1, el consumo cambia X." Esto le permite aprender el impacto real de cada día sin asumir relaciones numéricas falsas.

Se drop_first=True simplemente para ahorrar una columna. Si todas las demás columnas de día están en 0, el modelo ya sabe por descarte que ese día debe ser el que falta (el "día base").

Aunque la variable Hour (0 a 23) tiene una naturaleza cíclica que idealmente debería manejarse con técnicas específicas, se decidió mantenerla en su formato numérico y escalado porque ya fue sometida al proceso de escalamiento (StandardScaler), al igual que las demás variables numéricas. Esto asegura que Hour tenga una media de 0 y una desviación estándar de 1, evitando que su magnitud afecte de forma desproporcionada a modelos basados en distancia y la alternativa, Codificación One-Hot, crearía 24 nuevas columnas que requeririan un mayor costo computacional y harían más complicado el análisis.

##**5. Conclusiones**

**Imputación**: Se verificó la integridad de los datos, confirmando la ausencia de valores nulos en las 8 columnas. Esto reafirma la alta calidad de la recolección original por sensores y elimina la necesidad de técnicas de imputación.

**Escalamiento:** La aplicación de StandardScaler a las variables numéricas continuas  logró la estandarización ($\mu=0, \sigma=1$). Esto es importante para disminuir el sesgo introducido por las diferentes escalas de medición, preparando el dataset para modelos basados en la distancia.

**Variables Categóricas (Day):** La columna Day fue tratada mediante Codificación One-Hot. Esta técnica transforma la variable categórica nominal en un conjunto de variables binarias. Esto permite a los modelos capturar el impacto diferencial del día de la semana sobre el consumo de energía sin imponer una falsa relación ordinal.

**Decisión sobre Hour:** Se optó por mantener la variable Hour en su formato numérico escalado, evitando el aumento de la dimensionalidad (24 columnas) asociado al One-Hot Encoding.